# Modeling & Evaluation: Nuremberg Land Cover Change

**Two prediction tasks:**
- **Task A** -- Composition Prediction: predict land-cover percentages 3 years ahead
- **Task B** -- Change Prediction: predict 1-year land-cover change (deltas)

**Four models:** Ridge Regression, Random Forest, Gradient Boosted Trees, MLP

**Evaluation:** Temporal holdout, spatial block CV, change-specific metrics, stress tests, uncertainty, interpretability

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import clone
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

BASE = "/Users/hayat/Desktop/Study TU/Mlproj"
FEAT = f"{BASE}/Data-Generation-Pipeline/Features"
DATA = f"{BASE}/data/ml_tables/ml_training_data_bundle_20260302_200300/data/training_data"
OUT  = f"{BASE}/outputs"
os.makedirs(OUT, exist_ok=True)
os.makedirs(f"{OUT}/models", exist_ok=True)

print("Setup complete.")

Setup complete.


In [ ]:
# ── Load feature CSVs ──
feat_2020 = pd.read_csv(f"{FEAT}/features_clean_2020_renamed.csv")
feat_2021 = pd.read_csv(f"{FEAT}/features_clean_2021_renamed.csv")
feat_diff = pd.read_csv(f"{FEAT}/features_clean_diff_renamed.csv")

# ── Load target CSVs ──
target_2020 = pd.read_csv(f"{DATA}/Composition_prediction_in_3_years/2020/grid_stats.csv")
target_2021 = pd.read_csv(f"{DATA}/Composition_prediction_in_3_years/2021/grid_stats.csv")
target_diff = pd.read_csv(f"{DATA}/Composition_diff_in_one_year/grid_stats.csv")

# ── Strip year suffixes so all feature sets share common column names ──
def strip_suffix(df, suffix):
    return df.rename(columns={c: c.replace(f'_{suffix}', '') for c in df.columns if f'_{suffix}' in c})

feat_2020 = strip_suffix(feat_2020, '2020')
feat_2021 = strip_suffix(feat_2021, '2021')
feat_diff = strip_suffix(feat_diff, 'clean')

# ── Merge features with targets ──
target_cols_comp = ['cell_id', 'min_lon', 'min_lat', 'max_lon', 'max_lat',
                    'Built-up %', 'Permanent water bodies %', 'Other %']

target_cols_diff = ['cell_id', 'min_lon', 'min_lat', 'max_lon', 'max_lat',
                    'Built-up Baseline %', 'Permanent water bodies Baseline %',
                    'delta Built-up %', 'delta Permanent water bodies %', 'delta Other %']

df_2020 = feat_2020.merge(target_2020[target_cols_comp], on='cell_id')
df_2021 = feat_2021.merge(target_2021[target_cols_comp], on='cell_id')
df_diff = feat_diff.merge(target_diff[target_cols_diff], on='cell_id')

print(f"2020 composition dataset: {df_2020.shape}")
print(f"2021 composition dataset: {df_2021.shape}")
print(f"Diff (change) dataset:    {df_diff.shape}")
df_2020.head(3)

2020 composition dataset: (1600, 28)
2021 composition dataset: (1600, 28)
Diff (change) dataset:    (1600, 30)


,cell_id,row,col,R_mean,R_std,G_mean,G_std,B_mean,B_std,NDVI_1_mean,...,SWIR_2_std,SWIR_3_mean,SWIR_3_std,min_lon,min_lat,max_lon,max_lat,Built-up %,Permanent water bodies %,Other %
0,0,0,0,206.748596,54.325974,207.444702,52.221867,196.628006,60.284481,235.663193,...,7.154885,207.071899,54.168457,10.944175,49.549852,10.953158,49.55568,9.30,0.13,90.57
1,1,0,1,206.933701,57.237186,204.888901,57.748791,193.718903,64.707237,242.269806,...,8.152340,207.253799,57.057487,10.953158,49.549852,10.962141,49.55568,10.54,0.81,88.65
2,2,0,2,128.336502,63.341843,128.919403,63.391739,113.314499,69.000778,205.833694,...,36.463028,128.818207,63.281448,10.962141,49.549852,10.971125,49.55568,1.90,7.28,90.82


## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Target distributions for 2020
for i, col in enumerate(['Built-up %', 'Permanent water bodies %', 'Other %']):
    axes[0, i].hist(df_2020[col], bins=40, alpha=0.7, label='2020', color='steelblue')
    axes[0, i].hist(df_2021[col], bins=40, alpha=0.5, label='2021', color='coral')
    axes[0, i].set_title(col)
    axes[0, i].legend()
axes[0, 0].set_ylabel('Composition targets')

# Delta distributions
for i, col in enumerate(['delta Built-up %', 'delta Permanent water bodies %', 'delta Other %']):
    axes[1, i].hist(df_diff[col], bins=40, color='seagreen', alpha=0.7)
    axes[1, i].axvline(0, color='red', linestyle='--', alpha=0.5)
    axes[1, i].set_title(col)
axes[1, 0].set_ylabel('Change targets')

plt.tight_layout()
plt.show()

# Quick stats on changes
print("Change target statistics:")
print(df_diff[['delta Built-up %', 'delta Permanent water bodies %', 'delta Other %']].describe().round(3))

Change target statistics:
       delta Built-up %  delta Permanent water bodies %  delta Other %
count          1600.000                        1600.000       1600.000
mean              0.513                           0.005         -0.518
std               1.401                           0.374          1.386
min             -13.620                          -2.310        -10.100
25%              -0.050                           0.000         -0.830
50%               0.000                           0.000          0.000
75%               0.772                           0.000          0.090
max              10.020                          11.490          4.140


In [ ]:
# Check for zero-variance features (NDVI_3, SWIR_3)
spectral = ['R_mean', 'R_std', 'G_mean', 'G_std', 'B_mean', 'B_std',
            'NDVI_1_mean', 'NDVI_1_std', 'NDVI_2_mean', 'NDVI_2_std',
            'NDVI_3_mean', 'NDVI_3_std', 'SWIR_1_mean', 'SWIR_1_std',
            'SWIR_2_mean', 'SWIR_2_std', 'SWIR_3_mean', 'SWIR_3_std']

print("Zero-variance check (2020 dataset):")
for c in spectral:
    if df_2020[c].std() < 1e-6:
        print(f"  {c}: ALL ZEROS -- will drop")

# Correlation heatmap of spectral features
fig, ax = plt.subplots(figsize=(12, 10))
corr = df_2020[spectral].corr()
sns.heatmap(corr, annot=False, cmap='RdBu_r', center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix (2020 dataset)')
plt.tight_layout()
plt.show()

Zero-variance check (2020 dataset):
  NDVI_3_mean: ALL ZEROS -- will drop
  NDVI_3_std: ALL ZEROS -- will drop


In [ ]:
# Spatial pattern of Built-up % (2020)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, title in [(axes[0], df_2020, '2020 Built-up %'), (axes[1], df_2021, '2021 Built-up %')]:
    pivot = df.pivot(index='row', columns='col', values='Built-up %')
    im = ax.imshow(pivot, cmap='YlOrRd', aspect='equal')
    ax.set_title(title)
    ax.set_xlabel('col')
    ax.set_ylabel('row')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## Feature Engineering & Model Setup

In [ ]:
# ── Feature engineering ──
def engineer_features(df):
    df = df.copy()
    df['lat'] = (df['min_lat'] + df['max_lat']) / 2
    df['lon'] = (df['min_lon'] + df['max_lon']) / 2
    df['NDVI_veg_ratio'] = df['NDVI_1_mean'] / (df['R_mean'] + 1e-6)
    df['SWIR_moisture_ratio'] = df['SWIR_1_mean'] / (df['G_mean'] + 1e-6)
    return df

df_2020 = engineer_features(df_2020)
df_2021 = engineer_features(df_2021)
df_diff = engineer_features(df_diff)

# ── Drop zero-variance spectral features ──
SPECTRAL = [c for c in spectral if df_2020[c].std() > 1e-6]
dropped = set(spectral) - set(SPECTRAL)
if dropped:
    print(f"Dropped zero-variance features: {dropped}")

# ── Define feature lists ──
FEATURES_COMP = SPECTRAL + ['lat', 'lon', 'NDVI_veg_ratio', 'SWIR_moisture_ratio']
FEATURES_DIFF = FEATURES_COMP + ['Built-up Baseline %', 'Permanent water bodies Baseline %']

# Predict only 2 independent targets; derive Other = 100 - buildup - water
TARGETS_COMP = ['Built-up %', 'Permanent water bodies %']
TARGETS_DIFF = ['delta Built-up %', 'delta Permanent water bodies %']

print(f"Composition features ({len(FEATURES_COMP)}): {FEATURES_COMP}")
print(f"\nChange features ({len(FEATURES_DIFF)}): {FEATURES_DIFF}")
print(f"\nComposition targets: {TARGETS_COMP}")
print(f"Change targets: {TARGETS_DIFF}")

Dropped zero-variance features: {'NDVI_3_mean', 'NDVI_3_std'}
Composition features (20): ['R_mean', 'R_std', 'G_mean', 'G_std', 'B_mean', 'B_std', 'NDVI_1_mean', 'NDVI_1_std', 'NDVI_2_mean', 'NDVI_2_std', 'SWIR_1_mean', 'SWIR_1_std', 'SWIR_2_mean', 'SWIR_2_std', 'SWIR_3_mean', 'SWIR_3_std', 'lat', 'lon', 'NDVI_veg_ratio', 'SWIR_moisture_ratio']

Change features (22): ['R_mean', 'R_std', 'G_mean', 'G_std', 'B_mean', 'B_std', 'NDVI_1_mean', 'NDVI_1_std', 'NDVI_2_mean', 'NDVI_2_std', 'SWIR_1_mean', 'SWIR_1_std', 'SWIR_2_mean', 'SWIR_2_std', 'SWIR_3_mean', 'SWIR_3_std', 'lat', 'lon', 'NDVI_veg_ratio', 'SWIR_moisture_ratio', 'Built-up Baseline %', 'Permanent water bodies Baseline %']

Composition targets: ['Built-up %', 'Permanent water bodies %']
Change targets: ['delta Built-up %', 'delta Permanent water bodies %']


In [ ]:
# ── Model definitions ──
def build_models():
    return {
        'Ridge': make_pipeline(
            StandardScaler(),
            MultiOutputRegressor(Ridge(alpha=1.0))),
        'Random Forest': MultiOutputRegressor(
            RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)),
        'Gradient Boosting': MultiOutputRegressor(
            GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)),
        'MLP': make_pipeline(
            StandardScaler(),
            MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500,
                         early_stopping=True, random_state=42)),
    }

# ── Evaluation helper ──
def evaluate(y_true, y_pred, target_names):
    rows = []
    for i, name in enumerate(target_names):
        rows.append({
            'Target': name,
            'MAE': mean_absolute_error(y_true[:, i], y_pred[:, i]),
            'RMSE': np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])),
            'R2': r2_score(y_true[:, i], y_pred[:, i]),
        })
    return pd.DataFrame(rows)

# ── Train all models, return fitted models + predictions + metrics ──
def train_all(X_train, y_train, X_test, y_test, target_names, clip_lo=0, clip_hi=100):
    models = build_models()
    all_metrics = []
    all_preds = {}

    for name, model in models.items():
        print(f"  Training {name}...")
        model.fit(X_train, y_train)
        y_pred = np.clip(model.predict(X_test), clip_lo, clip_hi)
        metrics = evaluate(y_test, y_pred, target_names)
        metrics['Model'] = name
        all_metrics.append(metrics)
        all_preds[name] = y_pred

    metrics_df = pd.concat(all_metrics, ignore_index=True)
    return models, all_preds, metrics_df

print("Model helpers ready.")

Model helpers ready.


## Task A: Composition Prediction

**Phase 1 -- Bidirectional Temporal Holdout**
- Fold 1: Train on 2020 data (2017 Sentinel → 2020 labels), test on 2021 data (2018 Sentinel → 2021 labels)
- Fold 2: Train on 2021 data, test on 2020 data
- Report average metrics across both folds

In [ ]:
# ── Prepare data ──
X_2020 = df_2020[FEATURES_COMP].values
y_2020 = df_2020[TARGETS_COMP].values
X_2021 = df_2021[FEATURES_COMP].values
y_2021 = df_2021[TARGETS_COMP].values

# ── Fold 1: Train 2020, Test 2021 ──
print("=== Fold 1: Train 2020 → Test 2021 ===")
models_f1, preds_f1, metrics_f1 = train_all(X_2020, y_2020, X_2021, y_2021, TARGETS_COMP)

# ── Fold 2: Train 2021, Test 2020 ──
print("\n=== Fold 2: Train 2021 → Test 2020 ===")
models_f2, preds_f2, metrics_f2 = train_all(X_2021, y_2021, X_2020, y_2020, TARGETS_COMP)

# ── Average metrics across both folds ──
avg = pd.concat([metrics_f1, metrics_f2]).groupby(['Model', 'Target'])[['MAE', 'RMSE', 'R2']].mean().round(4)
print("\n=== Task A: Bidirectional Temporal CV (Averaged) ===")
avg

=== Fold 1: Train 2020 → Test 2021 ===
  Training Ridge...
  Training Random Forest...


  Training Gradient Boosting...


  Training MLP...



=== Fold 2: Train 2021 → Test 2020 ===
  Training Ridge...
  Training Random Forest...


  Training Gradient Boosting...


  Training MLP...



=== Task A: Bidirectional Temporal CV (Averaged) ===


MAE    RMSE      R2
Model             Target                                          
Gradient Boosting Built-up %                5.3253  8.6832  0.8781
                  Permanent water bodies %  0.2954  0.8642  0.8949
MLP               Built-up %                4.6955  7.5164  0.9092
                  Permanent water bodies %  0.3768  0.9126  0.8829
Random Forest     Built-up %                5.6619  9.1507  0.8641
                  Permanent water bodies %  0.3239  0.9106  0.8836
Ridge             Built-up %                6.6061  9.7581  0.8463
                  Permanent water bodies %  0.7439  1.5504  0.6626

In [ ]:
# ── Summary comparison bar chart ──
summary = avg.reset_index()
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(['MAE', 'RMSE', 'R2']):
    pivot = summary.pivot(index='Model', columns='Target', values=metric)
    pivot.plot(kind='bar', ax=axes[i], rot=15)
    axes[i].set_title(metric)
    axes[i].legend(fontsize=8)
plt.suptitle('Task A: Composition Prediction – Model Comparison', fontsize=13)
plt.tight_layout()
plt.show()

# Pick best model (lowest average MAE)
best_name_a = summary.groupby('Model')['MAE'].mean().idxmin()
print(f"\nBest model for Task A: {best_name_a}")


Best model for Task A: MLP


### Phase 2 -- Combined Final Model (3200 cells)

Combine 2020 + 2021 datasets, retrain all models with spatial block cross-validation.
Train final model on all data for export.

In [ ]:
# ── Combine 2020 + 2021 ──
df_combined = pd.concat([df_2020, df_2021], ignore_index=True)
X_comb = df_combined[FEATURES_COMP].values
y_comb = df_combined[TARGETS_COMP].values

# Spatial block IDs: 4x4 grid of 10x10 cell blocks
df_combined['block_id'] = (df_combined['row'] // 10) * 4 + (df_combined['col'] // 10)
groups = df_combined['block_id'].values
gkf = GroupKFold(n_splits=4)

# ── Spatial block CV predictions for all 4 models ──
cv_preds_comp = {}
cv_metrics_comp = []

for model_name in ['Ridge', 'Random Forest', 'Gradient Boosting', 'MLP']:
    print(f"  Spatial CV for {model_name}...")
    model_template = build_models()[model_name]
    y_pred_cv = np.zeros_like(y_comb)

    for train_idx, test_idx in gkf.split(X_comb, y_comb, groups):
        m = clone(model_template)
        m.fit(X_comb[train_idx], y_comb[train_idx])
        y_pred_cv[test_idx] = np.clip(m.predict(X_comb[test_idx]), 0, 100)

    cv_preds_comp[model_name] = y_pred_cv
    metrics = evaluate(y_comb, y_pred_cv, TARGETS_COMP)
    metrics['Model'] = model_name
    cv_metrics_comp.append(metrics)

cv_metrics_comp_df = pd.concat(cv_metrics_comp, ignore_index=True)
print("\n=== Phase 2: Spatial Block CV on Combined 3200 cells ===")
cv_metrics_comp_df.pivot(index='Model', columns='Target', values='MAE').round(4)

  Spatial CV for Ridge...
  Spatial CV for Random Forest...


  Spatial CV for Gradient Boosting...


  Spatial CV for MLP...



=== Phase 2: Spatial Block CV on Combined 3200 cells ===


Target,Built-up %,Permanent water bodies %
Model,,
Gradient Boosting,4.5057,0.4377
MLP,4.3960,0.4620
Random Forest,4.5686,0.4169
Ridge,6.0504,0.7870


In [ ]:
# ── Train final models on ALL combined data (for export) ──
final_models_comp = {}
for model_name in ['Ridge', 'Random Forest', 'Gradient Boosting', 'MLP']:
    m = build_models()[model_name]
    m.fit(X_comb, y_comb)
    final_models_comp[model_name] = m
    print(f"  {model_name} trained on 3200 cells.")

print("Final composition models ready.")

  Ridge trained on 3200 cells.


  Random Forest trained on 3200 cells.


  Gradient Boosting trained on 3200 cells.


  MLP trained on 3200 cells.
Final composition models ready.


## Task B: Change Prediction (1-year delta)

Predict `delta Built-up %` and `delta Permanent water bodies %` using 2020 Sentinel features + baseline composition.
Evaluation via spatial block cross-validation (same 4x4 block scheme).

In [ ]:
# ── Prepare diff data ──
X_diff = df_diff[FEATURES_DIFF].values
y_diff = df_diff[TARGETS_DIFF].values

df_diff['block_id'] = (df_diff['row'] // 10) * 4 + (df_diff['col'] // 10)
groups_diff = df_diff['block_id'].values
gkf_diff = GroupKFold(n_splits=4)

# ── Spatial block CV for all 4 models ──
cv_preds_diff = {}
cv_metrics_diff = []

for model_name in ['Ridge', 'Random Forest', 'Gradient Boosting', 'MLP']:
    print(f"  Spatial CV for {model_name}...")
    model_template = build_models()[model_name]
    y_pred_cv = np.zeros_like(y_diff)

    for train_idx, test_idx in gkf_diff.split(X_diff, y_diff, groups_diff):
        m = clone(model_template)
        m.fit(X_diff[train_idx], y_diff[train_idx])
        y_pred_cv[test_idx] = np.clip(m.predict(X_diff[test_idx]), -100, 100)

    cv_preds_diff[model_name] = y_pred_cv
    metrics = evaluate(y_diff, y_pred_cv, TARGETS_DIFF)
    metrics['Model'] = model_name
    cv_metrics_diff.append(metrics)

cv_metrics_diff_df = pd.concat(cv_metrics_diff, ignore_index=True)
print("\n=== Task B: Change Prediction – Spatial Block CV ===")
cv_metrics_diff_df.pivot(index='Model', columns='Target', values=['MAE', 'R2']).round(4)

  Spatial CV for Ridge...
  Spatial CV for Random Forest...


  Spatial CV for Gradient Boosting...


  Spatial CV for MLP...



=== Task B: Change Prediction – Spatial Block CV ===


MAE                                 \
Target            delta Built-up % delta Permanent water bodies %   
Model                                                               
Gradient Boosting           0.6919                         0.1181   
MLP                         0.7491                         0.1600   
Random Forest               0.6559                         0.1122   
Ridge                       0.7724                         0.1189   

                                R2                                 
Target            delta Built-up % delta Permanent water bodies %  
Model                                                              
Gradient Boosting           0.2614                        -0.5172  
MLP                         0.2572                        -0.1864  
Random Forest               0.3078                        -0.2361  
Ridge                       0.1811                        -0.0456

In [ ]:
# ── Summary bar chart for Task B ──
summary_b = cv_metrics_diff_df
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(['MAE', 'RMSE', 'R2']):
    pivot = summary_b.pivot(index='Model', columns='Target', values=metric)
    pivot.plot(kind='bar', ax=axes[i], rot=15)
    axes[i].set_title(metric)
    axes[i].legend(fontsize=8)
plt.suptitle('Task B: Change Prediction – Model Comparison', fontsize=13)
plt.tight_layout()
plt.show()

best_name_b = summary_b.groupby('Model')['MAE'].mean().idxmin()
print(f"\nBest model for Task B: {best_name_b}")

# ── Train final change models on ALL diff data ──
final_models_diff = {}
for model_name in ['Ridge', 'Random Forest', 'Gradient Boosting', 'MLP']:
    m = build_models()[model_name]
    m.fit(X_diff, y_diff)
    final_models_diff[model_name] = m
print("Final change models trained.")


Best model for Task B: Random Forest


Final change models trained.


## Evaluation Beyond Accuracy

### Change-Specific Metrics
- **False Change Rate**: Among stable cells (|true delta| < 0.5%), how often does the model hallucinate change?
- **Change Direction Accuracy**: Among genuinely changing cells (|true delta| > 1%), does the model predict the right sign?

In [ ]:
change_metrics_rows = []

for model_name, y_pred in cv_preds_diff.items():
    for i, target in enumerate(TARGETS_DIFF):
        true = y_diff[:, i]
        pred = y_pred[:, i]

        # False Change Rate: stable cells (|true| < 0.5) predicted as changed (|pred| > 0.5)
        stable_mask = np.abs(true) < 0.5
        if stable_mask.sum() > 0:
            false_change = (np.abs(pred[stable_mask]) > 0.5).mean()
        else:
            false_change = np.nan

        # Change Direction Accuracy: cells with real change (|true| > 1)
        change_mask = np.abs(true) > 1.0
        if change_mask.sum() > 0:
            direction_acc = (np.sign(pred[change_mask]) == np.sign(true[change_mask])).mean()
        else:
            direction_acc = np.nan

        change_metrics_rows.append({
            'Model': model_name,
            'Target': target,
            'False Change Rate': round(false_change, 4),
            'Direction Accuracy': round(direction_acc, 4),
            'N_stable': stable_mask.sum(),
            'N_changed': change_mask.sum(),
        })

change_metrics_df = pd.DataFrame(change_metrics_rows)
print("=== Change-Specific Metrics ===")
change_metrics_df

=== Change-Specific Metrics ===


,Model,Target,False Change Rate,Direction Accuracy,N_stable,N_changed
0,Ridge,delta Built-up %,0.2755,0.8454,991,401
1,Ridge,delta Permanent water bodies %,0.0000,0.5714,1530,28
2,Random Forest,delta Built-up %,0.1352,0.8329,991,401
3,Random Forest,delta Permanent water bodies %,0.0176,0.4643,1530,28
4,Gradient Boosting,delta Built-up %,0.1625,0.8180,991,401
5,Gradient Boosting,delta Permanent water bodies %,0.0176,0.5357,1530,28
6,MLP,delta Built-up %,0.2583,0.8354,991,401
7,MLP,delta Permanent water bodies %,0.0085,0.6071,1530,28


### Stress Test 1: Synthetic Feature Noise

Train on clean data, test on data with increasing Gaussian noise added to features.
Simulates sensor noise, atmospheric interference, and cloud residuals.

In [ ]:
# Use Task A Fold 1 setup: train on 2020, test on 2021 with noise
noise_levels = [0, 0.05, 0.10, 0.20, 0.30, 0.50]
rng = np.random.RandomState(42)
feature_stds = X_2021.std(axis=0)

noise_results = []

for model_name in ['Ridge', 'Random Forest', 'Gradient Boosting', 'MLP']:
    model = clone(models_f1[model_name])
    model.fit(X_2020, y_2020)

    for sigma in noise_levels:
        X_noisy = X_2021 + rng.normal(0, sigma, X_2021.shape) * feature_stds
        y_pred = np.clip(model.predict(X_noisy), 0, 100)
        mae = mean_absolute_error(y_2021, y_pred)
        noise_results.append({'Model': model_name, 'Noise sigma': sigma, 'MAE': mae})

noise_df = pd.DataFrame(noise_results)

fig, ax = plt.subplots(figsize=(9, 5))
for model_name in noise_df['Model'].unique():
    sub = noise_df[noise_df['Model'] == model_name]
    ax.plot(sub['Noise sigma'], sub['MAE'], marker='o', label=model_name)
ax.set_xlabel('Noise level (fraction of feature std)')
ax.set_ylabel('MAE (percentage points)')
ax.set_title('Stress Test: Model Robustness to Feature Noise')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Stress Test 2: Feature Group Ablation

Zero out entire spectral groups (RGB, NDVI, SWIR) to see which bands the models depend on most.
Simulates data loss from cloud cover (optical) or sensor failure.

In [ ]:
feature_groups = {
    'RGB': [f for f in FEATURES_COMP if f.startswith(('R_', 'G_', 'B_'))],
    'NDVI': [f for f in FEATURES_COMP if f.startswith('NDVI')],
    'SWIR': [f for f in FEATURES_COMP if f.startswith('SWIR')],
}

ablation_rows = []
for model_name in ['Ridge', 'Random Forest', 'Gradient Boosting', 'MLP']:
    model = clone(models_f1[model_name])
    model.fit(X_2020, y_2020)

    # Baseline (no ablation)
    y_base = np.clip(model.predict(X_2021), 0, 100)
    base_mae = mean_absolute_error(y_2021, y_base)
    ablation_rows.append({'Model': model_name, 'Ablated': 'None (baseline)', 'MAE': base_mae})

    for group_name, group_cols in feature_groups.items():
        col_idxs = [FEATURES_COMP.index(c) for c in group_cols if c in FEATURES_COMP]
        X_ablated = X_2021.copy()
        X_ablated[:, col_idxs] = 0
        y_pred = np.clip(model.predict(X_ablated), 0, 100)
        mae = mean_absolute_error(y_2021, y_pred)
        ablation_rows.append({'Model': model_name, 'Ablated': group_name, 'MAE': mae})

ablation_df = pd.DataFrame(ablation_rows)
pivot = ablation_df.pivot(index='Model', columns='Ablated', values='MAE').round(4)
pivot = pivot[['None (baseline)', 'RGB', 'NDVI', 'SWIR']]

print("=== Feature Group Ablation (MAE when group zeroed out) ===")
display(pivot)

fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax, rot=15)
ax.set_ylabel('MAE (percentage points)')
ax.set_title('Feature Group Ablation: Impact on Prediction Accuracy')
ax.legend(title='Ablated group')
plt.tight_layout()
plt.show()

=== Feature Group Ablation (MAE when group zeroed out) ===


Ablated,None (baseline),RGB,NDVI,SWIR
Model,,,,
Gradient Boosting,3.0424,4.0397,7.6397,4.8762
MLP,2.6846,10.1189,8.5001,19.4867
Random Forest,3.2576,4.3394,7.9459,4.5941
Ridge,3.9030,9.1663,17.7478,20.5534


### Spatial Error Maps

Where does the model fail? Visualize per-cell absolute error on the grid.

In [ ]:
# Use best model's CV predictions on combined data (Task A)
best_pred_comp = cv_preds_comp[best_name_a]
df_combined['error_buildup'] = np.abs(y_comb[:, 0] - best_pred_comp[:, 0])
df_combined['error_water'] = np.abs(y_comb[:, 1] - best_pred_comp[:, 1])

# Use best model's CV predictions on diff data (Task B)
best_pred_diff = cv_preds_diff[best_name_b]
df_diff['error_delta_buildup'] = np.abs(y_diff[:, 0] - best_pred_diff[:, 0])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Composition error: Built-up
# Need to handle combined (has 2 rows per cell), average the errors
err_comp = df_combined.groupby(['row', 'col'])['error_buildup'].mean().reset_index()
pivot1 = err_comp.pivot(index='row', columns='col', values='error_buildup')
im1 = axes[0].imshow(pivot1, cmap='Reds', aspect='equal')
axes[0].set_title(f'Task A: Built-up % error ({best_name_a})')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

err_water = df_combined.groupby(['row', 'col'])['error_water'].mean().reset_index()
pivot2 = err_water.pivot(index='row', columns='col', values='error_water')
im2 = axes[1].imshow(pivot2, cmap='Blues', aspect='equal')
axes[1].set_title(f'Task A: Water % error ({best_name_a})')
plt.colorbar(im2, ax=axes[1], shrink=0.8)

pivot3 = df_diff.pivot(index='row', columns='col', values='error_delta_buildup')
im3 = axes[2].imshow(pivot3, cmap='Oranges', aspect='equal')
axes[2].set_title(f'Task B: Δ Built-up error ({best_name_b})')
plt.colorbar(im3, ax=axes[2], shrink=0.8)

for ax in axes:
    ax.set_xlabel('col')
    ax.set_ylabel('row')
plt.tight_layout()
plt.show()

## Uncertainty Estimation

Using Random Forest tree-level variance as a proxy for prediction confidence.
High std across trees = low confidence (model is uncertain).

In [ ]:
# RF composition model: get individual tree predictions
rf_comp = final_models_comp['Random Forest']

# Predict with each tree for Built-up %
# MultiOutputRegressor wraps individual RandomForestRegressors
rf_buildup = rf_comp.estimators_[0]  # RF for Built-up %
rf_water = rf_comp.estimators_[1]    # RF for Water %

tree_preds_buildup = np.array([tree.predict(X_comb) for tree in rf_buildup.estimators_])
tree_preds_water = np.array([tree.predict(X_comb) for tree in rf_water.estimators_])

confidence_buildup = tree_preds_buildup.std(axis=0)  # high = uncertain
confidence_water = tree_preds_water.std(axis=0)

df_combined['confidence_buildup'] = confidence_buildup
df_combined['confidence_water'] = confidence_water

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_c1 = df_combined.groupby(['row', 'col'])['confidence_buildup'].mean().reset_index()
pivot_c1 = pivot_c1.pivot(index='row', columns='col', values='confidence_buildup')
im1 = axes[0].imshow(pivot_c1, cmap='YlOrRd', aspect='equal')
axes[0].set_title('Uncertainty: Built-up % (RF tree std)')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

pivot_c2 = df_combined.groupby(['row', 'col'])['confidence_water'].mean().reset_index()
pivot_c2 = pivot_c2.pivot(index='row', columns='col', values='confidence_water')
im2 = axes[1].imshow(pivot_c2, cmap='YlOrRd', aspect='equal')
axes[1].set_title('Uncertainty: Water % (RF tree std)')
plt.colorbar(im2, ax=axes[1], shrink=0.8)

for ax in axes:
    ax.set_xlabel('col')
    ax.set_ylabel('row')
plt.tight_layout()
plt.show()

print(f"Mean uncertainty Built-up: {confidence_buildup.mean():.2f}")
print(f"Mean uncertainty Water: {confidence_water.mean():.2f}")

Mean uncertainty Built-up: 3.88
Mean uncertainty Water: 0.27


## Interpretability

### Ridge Coefficients
Direct interpretation: each coefficient shows how a 1-unit change in a feature shifts the predicted percentage.

### Feature Importance (Random Forest & Gradient Boosting)
Tree-based importance shows which features the models rely on most.

### Helpful vs Misleading Explanations

In [ ]:
# ── Ridge Coefficients ──
ridge_model = final_models_comp['Ridge']
# Extract coefficients from pipeline: StandardScaler -> MultiOutputRegressor(Ridge)
ridge_multi = ridge_model.named_steps['multioutputregressor']
coefs = np.array([est.coef_ for est in ridge_multi.estimators_])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for i, target in enumerate(TARGETS_COMP):
    ax = axes[i]
    sorted_idx = np.argsort(np.abs(coefs[i]))[::-1]
    ax.barh([FEATURES_COMP[j] for j in sorted_idx], coefs[i][sorted_idx])
    ax.set_title(f'Ridge Coefficients: {target}')
    ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ── Tree-based Feature Importance ──
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for col_idx, (model_name, model) in enumerate([
    ('Random Forest', final_models_comp['Random Forest']),
    ('Gradient Boosting', final_models_comp['Gradient Boosting'])
]):
    for row_idx, (target_idx, target) in enumerate([(0, 'Built-up %'), (1, 'Water %')]):
        ax = axes[row_idx, col_idx]
        imp = model.estimators_[target_idx].feature_importances_
        sorted_idx = np.argsort(imp)[::-1][:12]
        ax.barh([FEATURES_COMP[j] for j in sorted_idx], imp[sorted_idx])
        ax.set_title(f'{model_name}: {target}')
        ax.invert_yaxis()

plt.suptitle('Feature Importance (Top 12)', fontsize=14)
plt.tight_layout()
plt.show()

### One Helpful Explanation

**"Cells with high NDVI values consistently have low built-up percentages."**

This aligns with domain knowledge: NDVI measures vegetation health, and heavily vegetated areas are not urbanized. A city planner can trust this signal -- if a cell's NDVI drops over time, it likely indicates vegetation loss and potential urbanization.

### One Misleading Explanation

**"The model identifies water body percentage as a strong predictor of built-up change."**

This could mislead a non-expert into thinking proximity to water drives construction. In reality, both Built-up % and Water % are small residual classes -- when one increases, the other may decrease purely because they both compete against the dominant "Other" class (vegetation, cropland). This is a statistical artifact of the compositional constraint, not a causal relationship.

## Where and Why the Model Is Likely Wrong

1. **Near-zero change dominance**: The vast majority of cells have delta values near zero. Models tend to default to "predict no change" as the safe option, potentially missing real but small transitions.

2. **Transition zones**: Cells at the urban-rural boundary have heterogeneous land cover that changes rapidly. These cells are hardest to predict because neighboring cells within the same spatial block can have very different compositions.

3. **Label noise**: ESA WorldCover has inherent classification errors at class boundaries. The model inherits these errors as "ground truth." The denoising applied to the 2020 labels (removing grass-to-building misclassifications) partially mitigates this but doesn't eliminate it.

4. **Seasonal water**: Small water bodies (ponds, seasonal streams) can appear or disappear between years due to seasonal variation, not actual land-cover change. The model may interpret this as real change.

5. **Temporal mismatch**: Sentinel-2 features are summer composites (June-September), while ESA WorldCover is an annual product. The exact dates of the imagery used for classification may not align, introducing noise.

6. **Single prediction horizon**: We only tested a 3-year prediction gap. Model accuracy at shorter or longer horizons is unknown and likely different.

## Export for Dashboard Team

Save all predictions, confidence scores, and trained models.
The dashboard team loads these CSVs directly -- each row has predictions from all 4 models.

In [ ]:
# ── Composition predictions (Phase 2 CV predictions, all 4 models) ──
comp_export = df_combined[['cell_id', 'row', 'col', 'lat', 'lon',
                            'Built-up %', 'Permanent water bodies %']].copy()
comp_export['Other %'] = 100 - comp_export['Built-up %'] - comp_export['Permanent water bodies %']

for model_name, preds in cv_preds_comp.items():
    tag = model_name.lower().replace(' ', '_')
    comp_export[f'{tag}_buildup'] = preds[:, 0]
    comp_export[f'{tag}_water'] = preds[:, 1]
    comp_export[f'{tag}_other'] = 100 - preds[:, 0] - preds[:, 1]

comp_export['rf_confidence_buildup'] = df_combined['confidence_buildup'].values
comp_export['rf_confidence_water'] = df_combined['confidence_water'].values

comp_export.to_csv(f"{OUT}/composition_predictions.csv", index=False)
print(f"Saved: {OUT}/composition_predictions.csv  ({comp_export.shape})")

# ── Change predictions (CV predictions, all 4 models) ──
diff_export = df_diff[['cell_id', 'row', 'col', 'lat', 'lon',
                        'delta Built-up %', 'delta Permanent water bodies %']].copy()
diff_export['delta Other %'] = -(diff_export['delta Built-up %'] + diff_export['delta Permanent water bodies %'])

for model_name, preds in cv_preds_diff.items():
    tag = model_name.lower().replace(' ', '_')
    diff_export[f'{tag}_delta_buildup'] = preds[:, 0]
    diff_export[f'{tag}_delta_water'] = preds[:, 1]
    diff_export[f'{tag}_delta_other'] = -(preds[:, 0] + preds[:, 1])

diff_export.to_csv(f"{OUT}/change_predictions.csv", index=False)
print(f"Saved: {OUT}/change_predictions.csv  ({diff_export.shape})")

# ── Evaluation summary ──
eval_summary = pd.concat([
    cv_metrics_comp_df.assign(Task='Composition'),
    cv_metrics_diff_df.assign(Task='Change')
], ignore_index=True)
eval_summary.to_csv(f"{OUT}/evaluation_summary.csv", index=False)
print(f"Saved: {OUT}/evaluation_summary.csv")

# ── Save trained models ──
for name, model in final_models_comp.items():
    path = f"{OUT}/models/composition_{name.lower().replace(' ', '_')}.joblib"
    joblib.dump(model, path)
for name, model in final_models_diff.items():
    path = f"{OUT}/models/change_{name.lower().replace(' ', '_')}.joblib"
    joblib.dump(model, path)

print(f"\nAll models saved to {OUT}/models/")
print("Export complete. Dashboard team can load CSVs directly.")

Saved: /Users/hayat/Desktop/Study TU/Mlproj/outputs/composition_predictions.csv  ((3200, 22))
Saved: /Users/hayat/Desktop/Study TU/Mlproj/outputs/change_predictions.csv  ((1600, 20))
Saved: /Users/hayat/Desktop/Study TU/Mlproj/outputs/evaluation_summary.csv

All models saved to /Users/hayat/Desktop/Study TU/Mlproj/outputs/models/
Export complete. Dashboard team can load CSVs directly.
